In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.6 MB/s eta 0:00:00


In [ ]:
!pip install filterpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=4336c68ca2809dcc0788f7a314d3025b59004a3382c3d866b3ca447964fd0118
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy


In [ ]:
from ultralytics import YOLO
from google.colab import drive
import os, glob
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from collections import defaultdict
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
drive.mount('/content/drive')
drive_folder = '/content/drive/MyDrive/CV_Final_Project'
print("Extracting sequences from permanent Google Drive storage...")
for seq in ['01', '02', '03', '04']:
    os.system(f'unzip -q "{drive_folder}/{seq}.zip" -d /content/sequence_{seq}')
print("All zip files successfully unzipped and ready for tracking processing!")

Mounted at /content/drive
Extracting sequences from permanent Google Drive storage...
All zip files successfully unzipped and ready for tracking processing!


In [ ]:
CONFIG = {
    # Detector
    "MODEL_WEIGHTS": "yolov8x.pt",
    "IMG_SIZE": 1280,
    "DET_NMS_IOU": 0.7,          # Ultralytics default.
    "USE_TILED_INFERENCE": False,  # see detect_frame() below -- catches small
                                    # far-field pedestrians but ~4-5x slower

    # ByteTrack-style confidence split (first chode values of the ByteTrack paper's
    # published defaults, then optimised according to tracking video )
    "HIGH_THRESH": 0.35,          # >= this: stage-1 (IoU + gated appearance)
    "LOW_THRESH": 0.1,           # [LOW_THRESH, HIGH_THRESH): stage-2 rescue
                                  # only (IoU only, can't start new tracks)
    "NEW_TRACK_THRESH": 0.6,     # only spawn a new track from a detection at
                                  # least this confident

    # Tracker (unchanged from original)
    "MAX_AGE": 30,
    "MIN_HITS": 3,
    "IOU_THRESHOLD": 0.3,

    # DeepSORT appearance gate: chi-squared 95th percentile for 4 DOF.
    # This is a textbook constant (Wojke et al. 2017), not a free parameter.
    "GATING_THRESHOLD": 9.4877,
    "MAX_COSINE_DIST": 0.25,
    "APPEARANCE_WEIGHT": 0.6,    # weight on appearance vs (1-w) on IoU,
                                  # stage-1 cost only

    # Re-ID backbone
    "USE_TORCHREID_OSNET": False,
    "OSNET_WEIGHTS_PATH": None,
}


In [ ]:
class GeMPooling(nn.Module):
    """Generalized-Mean pooling: interpolates between average-pool (p=1) and
    max-pool (p->inf). Outperforms plain average pooling for retrieval-style
    embeddings (Radenovic et al., 'Fine-tuning CNN Image Retrieval with No
    Human Annotation', TPAMI 2018) -- relevant here because Re-ID is a
    retrieval problem, not a classification one."""
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = p
        self.eps = eps

    def forward(self, x):
        x = x.clamp(min=self.eps).pow(self.p)
        x = F.adaptive_avg_pool2d(x, 1)
        return x.pow(1.0 / self.p)

In [ ]:
class ReIDExtractor(nn.Module):
    """
    Default appearance embedder: ResNet50 (ImageNet-pretrained, 2048-d --
    much richer than MobileNetV3-Small's 576-d), classifier head removed,
    GeM pooling instead of average pooling, L2-normalized, fused with a
    96-d HSV color histogram.

    """
    def __init__(self):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(*list(backbone.children())[:-2])  # drop avgpool + fc
        self.pool = GeMPooling(p=3.0)
        self.eval()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.to(self.device)

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((256, 128)),   # standard Re-ID aspect ratio --
                                               # taller than the 128x64 used
                                               # before, keeps more of the body
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225]),
        ])
        self.embed_dim = 2048 + 96

    @staticmethod
    def _color_hist(crop):
        """32-bin HSV histogram per channel, L2-normalized. Cheap and
        complements the CNN embedding when clothing color is the most
        reliable cue (e.g. one bright jacket in a crowd of similarly-shaped
        people)."""
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        hist = [cv2.calcHist([hsv], [ch], None, [32], [0, 256]).flatten() for ch in range(3)]
        hist = np.concatenate(hist).astype(np.float32)
        norm = np.linalg.norm(hist)
        return hist / (norm + 1e-6) if norm > 0 else hist

    @torch.no_grad()
    def extract_batch(self, crops):
        """Extract embeddings for every crop in a frame as ONE batched
        forward pass, instead of a Python loop over 246 individual crops --
        meaningfully faster this week."""
        valid_idx = [i for i, c in enumerate(crops) if c.size > 0 and c.shape[0] >= 5 and c.shape[1] >= 5]
        out = np.zeros((len(crops), self.embed_dim), dtype=np.float32)
        if not valid_idx:
            return out

        tensors = torch.stack([self.transform(crops[i]) for i in valid_idx]).to(self.device)
        feats = self.pool(self.stem(tensors))
        feats = feats.view(feats.size(0), -1).cpu().numpy()
        feats = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-6)

        for k, i in enumerate(valid_idx):
            color = self._color_hist(crops[i])
            combined = np.concatenate([feats[k], color])
            norm = np.linalg.norm(combined)
            out[i] = combined / norm if norm > 0 else combined
        return out

In [ ]:
class ReIDExtractorOSNet:
    """
    OPTIONAL stronger alternative: true person-Re-ID embeddings, OSNet x1.0
    trained on MSMT17 -- what the strongest Week 4 submission used. Requires:
        pip install torchreid
    and manually downloading the MSMT17-pretrained OSNet checkpoint from the
    torchreid model zoo (Google-Drive hosted -- occasionally rate-limited,
    which is the only reason this isn't the default for a graded submission).
    Point CONFIG["OSNET_WEIGHTS_PATH"] at the .pth file and set
    CONFIG["USE_TORCHREID_OSNET"] = True to use it.
    """
    def __init__(self, weights_path, device=None):
        from torchreid.utils import FeatureExtractor
        self.extractor = FeatureExtractor(
            model_name="osnet_x1_0",
            model_path=weights_path,
            device=device or ("cuda" if torch.cuda.is_available() else "cpu"),
        )
        self.embed_dim = 512

    def extract_batch(self, crops):
        valid_idx = [i for i, c in enumerate(crops) if c.size > 0 and c.shape[0] >= 5 and c.shape[1] >= 5]
        out = np.zeros((len(crops), self.embed_dim), dtype=np.float32)
        if not valid_idx:
            return out
        feats = self.extractor([crops[i] for i in valid_idx]).cpu().numpy()
        feats = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-6)
        for k, i in enumerate(valid_idx):
            out[i] = feats[k]
        return out

In [ ]:
def iou_batch(bb_test, bb_gt):
    bb_gt = np.expand_dims(bb_gt, 0)
    bb_test = np.expand_dims(bb_test, 1)
    xx1 = np.maximum(bb_test[..., 0], bb_gt[..., 0])
    yy1 = np.maximum(bb_test[..., 1], bb_gt[..., 1])
    xx2 = np.minimum(bb_test[..., 2], bb_gt[..., 2])
    yy2 = np.minimum(bb_test[..., 3], bb_gt[..., 3])
    w = np.maximum(0., xx2 - xx1)
    h = np.maximum(0., yy2 - yy1)
    wh = w * h
    o = wh / ((bb_test[..., 2] - bb_test[..., 0]) * (bb_test[..., 3] - bb_test[..., 1])
              + (bb_gt[..., 2] - bb_gt[..., 0]) * (bb_gt[..., 3] - bb_gt[..., 1]) - wh)
    return o

In [ ]:
class KalmanBoxTracker(object):
    count = 0

    def __init__(self, bbox, confidence=0.5):
        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        self.kf.F = np.array([[1,0,0,0,1,0,0],[0,1,0,0,0,1,0],[0,0,1,0,0,0,1],[0,0,0,1,0,0,0],
                               [0,0,0,0,1,0,0],[0,0,0,0,0,1,0],[0,0,0,0,0,0,1]])
        self.kf.H = np.array([[1,0,0,0,0,0,0],[0,1,0,0,0,0,0],[0,0,1,0,0,0,0],[0,0,0,1,0,0,0]])

        self.base_R = np.eye(4)
        self.base_R[2:, 2:] *= 10.0
        self.kf.P[4:, 4:] *= 1000.
        self.kf.P *= 10.
        self.kf.Q[-1, -1] *= 0.01
        self.kf.Q[4:, 4:] *= 0.01
        self.kf.x[:4] = self.convert_bbox_to_z(bbox)
        self.kf.R = self.base_R * (1.0 - confidence)  # NSA Kalman init

        self.time_since_update = 0
        self.id = KalmanBoxTracker.count
        KalmanBoxTracker.count += 1
        self.history = []
        self.hits = 0
        self.hit_streak = 0
        self.age = 0
        self.feature = None  # appearance embedding, set on creation/update

    def update(self, bbox, confidence=0.5):
        self.time_since_update = 0
        self.history = []
        self.hits += 1
        self.hit_streak += 1
        self.kf.R = self.base_R * (1.0 - confidence)  # NSA Kalman adaptive noise
        self.kf.update(self.convert_bbox_to_z(bbox))

    def predict(self):
        if (self.kf.x[6] + self.kf.x[2]) <= 0:
            self.kf.x[6] *= 0.0
        self.kf.predict()
        self.age += 1
        if self.time_since_update > 0:
            self.hit_streak = 0
        self.time_since_update += 1
        self.history.append(self.convert_x_to_bbox(self.kf.x))
        return self.history[-1]

    def get_state(self):
        return self.convert_x_to_bbox(self.kf.x)

    @staticmethod
    def convert_bbox_to_z(bbox):
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        x = bbox[0] + w / 2.
        y = bbox[1] + h / 2.
        s = w * h
        r = w / float(h) if h > 0 else 0
        return np.array([x, y, s, r]).reshape((4, 1))

    @staticmethod
    def convert_x_to_bbox(x, score=None):
        w = np.sqrt(np.maximum(0, x[2] * x[3]))
        h = x[2] / w if w > 0 else 1
        if score is None:
            return np.array([x[0]-w/2., x[1]-h/2., x[0]+w/2., x[1]+h/2.]).reshape((1, 4))
        return np.array([x[0]-w/2., x[1]-h/2., x[0]+w/2., x[1]+h/2., score]).reshape((1, 5))

In [ ]:
def gating_distance(tracker, bbox_xyxy):
    """Mahalanobis distance between a track's predicted state and a
    candidate detection, in the Kalman filter's own measurement space.
    This is the actual DeepSORT gate (Wojke et al., 2017): appearance
    matches are only trusted within this statistical test, not blended
    with unlimited weight the way a plain linear cost combination does.
    Uses `base_R` (nominal noise), not the confidence-scaled NSA `kf.R`,
    since gating should be a fixed statistical test independent of the
    candidate detection's own confidence."""
    z = KalmanBoxTracker.convert_bbox_to_z(bbox_xyxy).reshape(-1)
    x = tracker.kf.x[:4].reshape(-1)
    H = tracker.kf.H
    S = H @ tracker.kf.P @ H.T + tracker.base_R
    diff = z - x
    try:
        S_inv = np.linalg.inv(S)
    except np.linalg.LinAlgError:
        return 1e6
    return float(diff.T @ S_inv @ diff)

In [ ]:
def build_cost_matrix(dets_xyxy, det_feats, tracks, cfg):
    """Stage-1 cost: IoU + appearance, both used as GATES (hard rejection)
    rather than blended without limit -- a candidate pair is only eligible
    if it passes the IoU gate, the Mahalanobis gate, AND the cosine-distance
    gate. Only pairs that pass all three get a real (low) cost; everything
    else gets a large placeholder cost so the Hungarian solver effectively
    never picks it unless there's truly no better option."""
    n_d, n_t = len(dets_xyxy), len(tracks)
    if n_t == 0 or n_d == 0:
        return np.empty((n_d, n_t))
    dets_arr = np.array(dets_xyxy)
    tracks_arr = np.array([t.get_state()[0][:4] for t in tracks])
    iou_mat = iou_batch(dets_arr, tracks_arr)
    cost = np.full((n_d, n_t), 1e5, dtype=np.float64)

    for ti, trk in enumerate(tracks):
        for di in range(n_d):
            if iou_mat[di, ti] < cfg["IOU_THRESHOLD"]:
                continue
            if gating_distance(trk, dets_arr[di]) > cfg["GATING_THRESHOLD"]:
                continue
            cos_dist = 1.0 - float(np.dot(det_feats[di], trk.feature)) if trk.feature is not None else 1.0
            if cos_dist > cfg["MAX_COSINE_DIST"]:
                continue
            iou_cost = 1.0 - iou_mat[di, ti]
            cost[di, ti] = cfg["APPEARANCE_WEIGHT"] * cos_dist + (1 - cfg["APPEARANCE_WEIGHT"]) * iou_cost
    return cost

In [ ]:
def matching_cascade(dets_xyxy, det_feats, tracks, cfg):
    """Real DeepSORT matching cascade: tracks that were updated most
    recently (time_since_update == 1) get first pick of detections; only
    leftover detections trickle down to older, less-certain tracks. This
    stops a long-unconfirmed track from "stealing" a detection that
    actually belongs to a track seen one frame ago."""
    unmatched_dets = list(range(len(dets_xyxy)))
    matches = []
    remaining_tracks = list(range(len(tracks)))

    for age in range(1, cfg["MAX_AGE"] + 1):
        if not unmatched_dets or not remaining_tracks:
            break
        tier_tracks = [i for i in remaining_tracks if tracks[i].time_since_update == age]
        if not tier_tracks:
            continue
        sub_dets = [dets_xyxy[d] for d in unmatched_dets]
        sub_feats = [det_feats[d] for d in unmatched_dets]
        sub_tracks = [tracks[i] for i in tier_tracks]
        cost = build_cost_matrix(sub_dets, sub_feats, sub_tracks, cfg)
        if cost.size == 0:
            continue
        row_ind, col_ind = linear_sum_assignment(cost)
        newly_matched_dets, newly_matched_tracks = [], []
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] >= 1e5:
                continue
            matches.append((unmatched_dets[r], tier_tracks[c]))
            newly_matched_dets.append(unmatched_dets[r])
            newly_matched_tracks.append(tier_tracks[c])
        unmatched_dets = [d for d in unmatched_dets if d not in newly_matched_dets]
        remaining_tracks = [t for t in remaining_tracks if t not in newly_matched_tracks]

    return matches, unmatched_dets, remaining_tracks

In [ ]:
def iou_only_match(dets_xyxy, det_indices, tracks, track_indices, iou_threshold):
    """Plain IoU Hungarian match -- used both as the stage-1 fallback (for
    tracks/detections the appearance cascade couldn't resolve, mirroring
    DeepSORT's own IoU fallback for unconfirmed tracks) and as the
    ByteTrack stage-2 rescue pass against low-confidence detections."""
    if not det_indices or not track_indices:
        return [], det_indices, track_indices
    sub_dets = np.array([dets_xyxy[d] for d in det_indices])
    sub_tracks = np.array([tracks[t].get_state()[0][:4] for t in track_indices])
    iou_mat = iou_batch(sub_dets, sub_tracks)
    cost = 1.0 - iou_mat
    row_ind, col_ind = linear_sum_assignment(cost)
    matches, matched_d, matched_t = [], [], []
    for r, c in zip(row_ind, col_ind):
        if iou_mat[r, c] < iou_threshold:
            continue
        matches.append((det_indices[r], track_indices[c]))
        matched_d.append(det_indices[r])
        matched_t.append(track_indices[c])
    unmatched_d = [d for d in det_indices if d not in matched_d]
    unmatched_t = [t for t in track_indices if t not in matched_t]
    return matches, unmatched_d, unmatched_t

In [ ]:
class HybridTracker:
    """DeepSORT-style Mahalanobis-gated appearance cascade (stage 1, high-
    confidence detections) + ByteTrack-style low-confidence rescue (stage 2,
    IoU only, never creates new tracks) + NSA Kalman. Feeds into your
    existing `interpolate_trajectories` post-processing unchanged."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.tracks = []
        self.frame_count = 0

    def step(self, dets, det_feats):
        """dets: array of [x1,y1,x2,y2,conf]; det_feats: matching list of
        appearance embeddings (same order as dets)."""
        cfg = self.cfg
        self.frame_count += 1

        to_del = []
        for i, trk in enumerate(self.tracks):
            pos = trk.predict()[0]
            if np.any(np.isnan(pos)):
                to_del.append(i)
        for i in reversed(to_del):
            self.tracks.pop(i)

        if len(dets) > 0:
            high_mask = dets[:, 4] >= cfg["HIGH_THRESH"]
            low_mask = (dets[:, 4] >= cfg["LOW_THRESH"]) & (~high_mask)
        else:
            high_mask = np.array([], dtype=bool)
            low_mask = np.array([], dtype=bool)

        high_idx = np.where(high_mask)[0].tolist()
        low_idx = np.where(low_mask)[0].tolist()
        dets_xyxy = dets[:, :4] if len(dets) else np.empty((0, 4))

        # Stage 1: high-confidence detections, gated appearance cascade
        matches, unmatched_high, unmatched_tracks = matching_cascade(
            [dets_xyxy[i] for i in high_idx], [det_feats[i] for i in high_idx],
            self.tracks, cfg
        )
        matches = [(high_idx[d], t) for d, t in matches]
        unmatched_high = [high_idx[d] for d in unmatched_high]

        # Stage 1b: plain IoU fallback for anything the cascade missed
        fb_matches, unmatched_high, unmatched_tracks = iou_only_match(
            dets_xyxy, unmatched_high, self.tracks, unmatched_tracks, cfg["IOU_THRESHOLD"]
        )
        matches += fb_matches

        # Stage 2 (ByteTrack): rescue remaining tracks with low-conf dets,
        # IoU only. Unmatched low-conf detections are simply discarded --
        # never spawn a new track from them.
        lc_matches, unmatched_low, unmatched_tracks = iou_only_match(
            dets_xyxy, low_idx, self.tracks, unmatched_tracks, cfg["IOU_THRESHOLD"]
        )
        matches += lc_matches

        for d, t in matches:
            conf = dets[d, 4]
            self.tracks[t].update(dets_xyxy[d], confidence=conf)
            if self.tracks[t].feature is not None:
                self.tracks[t].feature = 0.9 * self.tracks[t].feature + 0.1 * det_feats[d]
            else:
                self.tracks[t].feature = det_feats[d]
            norm = np.linalg.norm(self.tracks[t].feature)
            if norm > 0:
                self.tracks[t].feature = self.tracks[t].feature / norm

        # New tracks ONLY from strong, unmatched high-confidence detections
        for d in unmatched_high:
            if dets[d, 4] < cfg["NEW_TRACK_THRESH"]:
                continue
            trk = KalmanBoxTracker(dets_xyxy[d], confidence=dets[d, 4])
            trk.feature = det_feats[d]
            self.tracks.append(trk)

        ret = []
        for i in reversed(range(len(self.tracks))):
            trk = self.tracks[i]
            d = trk.get_state()[0]
            if trk.time_since_update < 1 and (trk.hit_streak >= cfg["MIN_HITS"] or self.frame_count <= cfg["MIN_HITS"]):
                ret.append(np.concatenate((d, [trk.id + 1])).reshape(1, -1))
            if trk.time_since_update > cfg["MAX_AGE"]:
                self.tracks.pop(i)

        return np.concatenate(ret) if ret else np.empty((0, 5))

In [ ]:
def _nms_merge(boxes, scores, iou_thr=0.6):
    idx = np.argsort(-scores)
    keep = []
    while len(idx) > 0:
        i = idx[0]
        keep.append(i)
        if len(idx) == 1:
            break
        rest = idx[1:]
        ious = iou_batch(boxes[i:i+1], boxes[rest])[0]
        idx = rest[ious < iou_thr]
    return keep

In [ ]:
def detect_frame(model, frame, cfg):
    h_img, w_img = frame.shape[:2]
    all_boxes, all_scores = [], []

    results = model.predict(frame, classes=[0], conf=cfg["LOW_THRESH"],
                             imgsz=cfg["IMG_SIZE"], iou=cfg["DET_NMS_IOU"],
                             verbose=False)[0]
    if len(results.boxes) > 0:
        all_boxes.append(results.boxes.xyxy.cpu().numpy())
        all_scores.append(results.boxes.conf.cpu().numpy())

    if cfg.get("USE_TILED_INFERENCE", False):
        # 4 overlapping quadrant tiles -- catches small far-field pedestrians
        # that get missed even at imgsz=1280 on a full 1920x1080 frame.
        # Roughly quadruples per-frame inference time -- only turn this on
        # if you've checked your hardest sequence and small/distant people
        # are the dominant miss type.
        overlap = 0.2
        tile_h = int(h_img * (0.5 + overlap))
        tile_w = int(w_img * (0.5 + overlap))
        origins = [(0, 0), (0, w_img - tile_w), (h_img - tile_h, 0), (h_img - tile_h, w_img - tile_w)]
        for oy, ox in origins:
            tile = frame[oy:oy + tile_h, ox:ox + tile_w]
            tres = model.predict(tile, classes=[0], conf=cfg["LOW_THRESH"],
                                  imgsz=cfg["IMG_SIZE"], iou=cfg["DET_NMS_IOU"],
                                  verbose=False)[0]
            if len(tres.boxes) > 0:
                tb = tres.boxes.xyxy.cpu().numpy()
                tb[:, [0, 2]] += ox
                tb[:, [1, 3]] += oy
                all_boxes.append(tb)
                all_scores.append(tres.boxes.conf.cpu().numpy())

    if not all_boxes:
        return np.empty((0, 4)), np.empty((0,))

    boxes = np.concatenate(all_boxes)
    scores = np.concatenate(all_scores)
    keep = _nms_merge(boxes, scores, iou_thr=cfg["DET_NMS_IOU"])
    return boxes[keep], scores[keep]

In [ ]:
def interpolate_trajectories(results_list, max_gap=30):
    tracks = defaultdict(list)
    for line in results_list:
        parts = line.split(',')
        tracks[int(parts[1])].append([int(parts[0]), float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])])

    final_lines = []
    for obj_id, det_list in tracks.items():
        det_list.sort(key=lambda x: x[0])
        for i in range(len(det_list) - 1):
            curr_frame, cx, cy, cw, ch = det_list[i]
            next_frame, nx, ny, nw, nh = det_list[i+1]
            final_lines.append(f"{curr_frame},{obj_id},{cx:.2f},{cy:.2f},{cw:.2f},{ch:.2f},-1,-1,-1,-1")
            gap = next_frame - curr_frame
            if 1 < gap <= max_gap:
                for step in range(1, gap):
                    f = curr_frame + step
                    alpha = step / gap
                    final_lines.append(f"{f},{obj_id},{cx + alpha*(nx-cx):.2f},{cy + alpha*(ny-cy):.2f},{cw + alpha*(nw-cw):.2f},{ch + alpha*(nh-ch):.2f},-1,-1,-1,-1")
        if det_list:
            final_lines.append(f"{det_list[-1][0]},{obj_id},{det_list[-1][1]:.2f},{det_list[-1][2]:.2f},{det_list[-1][3]:.2f},{det_list[-1][4]:.2f},-1,-1,-1,-1")
    final_lines.sort(key=lambda x: int(x.split(',')[0]))
    return final_lines

In [ ]:
def process_extracted_sequence(search_root, seq_id, output_filepath, cfg=CONFIG):
    img_search_pattern = os.path.join(search_root, "**", "img")
    img_dirs = [d for d in glob.glob(img_search_pattern, recursive=True) if os.path.isdir(d)]
    if not img_dirs:
        print(f"Error: Could not find 'img' directory under {search_root}")
        return
    img_dir = img_dirs[0]
    frame_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
    if not frame_paths:
        print(f"Error: No .jpg frames found inside {img_dir}")
        return
    print(f"Located target sequence path: {img_dir} ({len(frame_paths)} frames)")

    model = YOLO(cfg["MODEL_WEIGHTS"])
    if cfg["USE_TORCHREID_OSNET"]:
        reid_model = ReIDExtractorOSNet(cfg["OSNET_WEIGHTS_PATH"])
    else:
        reid_model = ReIDExtractor()

    KalmanBoxTracker.count = 0  # reset ID counter between sequences
    tracker = HybridTracker(cfg)

    raw_results = []
    for frame_idx, frame_path in enumerate(frame_paths, start=1):
        frame = cv2.imread(frame_path)
        if frame is None:
            continue
        h_img, w_img = frame.shape[:2]

        boxes, scores = detect_frame(model, frame, cfg)

        crops = []
        for box in boxes:
            x1, y1, x2, y2 = map(int, np.clip(box, 0, [w_img, h_img, w_img, h_img]))
            crops.append(frame[y1:y2, x1:x2])
        feats = reid_model.extract_batch(crops) if crops else np.empty((0, reid_model.embed_dim))

        dets = np.concatenate([boxes, scores.reshape(-1, 1)], axis=1) if len(boxes) else np.empty((0, 5))
        tracked = tracker.step(dets, list(feats))

        for obj in tracked:
            x1, y1, x2, y2, obj_id = obj
            w, h = x2 - x1, y2 - y1
            if w > 0 and h > 0:
                raw_results.append(f"{frame_idx},{int(obj_id)},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f}")

        if frame_idx % 200 == 0:
            print(f"  seq {seq_id}: frame {frame_idx}/{len(frame_paths)}, active tracks: {len(tracker.tracks)}")

    final_output = interpolate_trajectories(raw_results, max_gap=30)
    with open(output_filepath, "w") as f:
        f.write("\n".join(final_output) + "\n")
    print(f"Saved submission output file: {output_filepath}\n")


if __name__ == "__main__":
    base_extraction_root = "/content"
    target_sequences = {
        "01": os.path.join(base_extraction_root, "sequence_01"),
        "02": os.path.join(base_extraction_root, "sequence_02"),
        "03": os.path.join(base_extraction_root, "sequence_03"),
        "04": os.path.join(base_extraction_root, "sequence_04"),
    }
    for seq_id, extraction_dir in target_sequences.items():
        if os.path.exists(extraction_dir):
            print(f"Beginning Execution on Sequence {seq_id}...")
            process_extracted_sequence(extraction_dir, seq_id, f"{seq_id}.txt")
        else:
            print(f"Skipping Sequence {seq_id}: Extraction target path not found.")

Beginning Execution on Sequence 01...
Located target sequence path: /content/sequence_01/01/img (429 frames)
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 157MB/s]


  seq 01: frame 200/429, active tracks: 27
  seq 01: frame 400/429, active tracks: 25
Saved submission output file: 01.txt

Beginning Execution on Sequence 02...
Located target sequence path: /content/sequence_02/02/img (2782 frames)
  seq 02: frame 200/2782, active tracks: 28
  seq 02: frame 400/2782, active tracks: 29
  seq 02: frame 600/2782, active tracks: 18
  seq 02: frame 800/2782, active tracks: 19
  seq 02: frame 1000/2782, active tracks: 36
  seq 02: frame 1200/2782, active tracks: 36
  seq 02: frame 1400/2782, active tracks: 30
  seq 02: frame 1600/2782, active tracks: 28
  seq 02: frame 1800/2782, active tracks: 28
  seq 02: frame 2000/2782, active tracks: 25
  seq 02: frame 2200/2782, active tracks: 32
  seq 02: frame 2400/2782, active tracks: 29
  seq 02: frame 2600/2782, active tracks: 32
Saved submission output file: 02.txt

Beginning Execution on Sequence 03...
Located target sequence path: /content/sequence_03/03/img (2405 frames)
  seq 03: frame 200/2405, active trac

In [ ]:
import os
import glob
import cv2

def create_tracking_video(sequence_dir, tracking_txt, output_video_path, fps=30.0):
    """
    Reads a sequence of images (handling macOS nested folders dynamically)
    and a formatted tracking text file, draws bounding boxes and IDs,
    and saves it as an MP4 video.
    """
    # 1. Parse the tracking text file matching the strict MOT 1-based index template
    frame_data = {}
    if not os.path.exists(tracking_txt):
        print(f"Error: Could not find {tracking_txt}. Run the tracker first!")
        return

    print(f"Parsing tracking file: {tracking_txt}...")
    with open(tracking_txt, 'r') as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) < 6:
                continue

            # Extract 1-based frame, ID, and bbox coordinates
            frame_idx = int(parts[0])
            obj_id = int(parts[1])
            x, y, w, h = map(float, parts[2:6])

            if frame_idx not in frame_data:
                frame_data[frame_idx] = []
            frame_data[frame_idx].append((obj_id, x, y, w, h))

    # 2. Dynamic path resolution for the 'img' directory to fix macOS nested folders
    img_search_pattern = os.path.join(sequence_dir, "**", "img")
    img_dirs = [d for d in glob.glob(img_search_pattern, recursive=True) if os.path.isdir(d)]

    if not img_dirs:
        print(f"Error: Could not find 'img' directory under path: {sequence_dir}")
        return

    img_dir = img_dirs[0]
    image_paths = sorted(glob.glob(os.path.join(img_dir, '*.jpg')))

    if not image_paths:
        print(f"Error: No images found inside resolved path: {img_dir}")
        return

    print(f"Resolved sequence location: {img_dir} ({len(image_paths)} frames found)")

    # 3. Setup the OpenCV Video Writer
    first_frame = cv2.imread(image_paths[0])
    height, width, _ = first_frame.shape

    # 'mp4v' is an excellent universal codec for OpenCV within notebook workspaces
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    print(f"Generating video rendering: {output_video_path}...")

    # 4. Loop through images and draw the boxes matching 1-based indices
    for frame_idx, img_path in enumerate(image_paths, start=1):
        frame = cv2.imread(img_path)
        if frame is None:
            continue

        # Check if we have tracking data for this frame index
        if frame_idx in frame_data:
            for obj_id, x, y, w, h in frame_data[frame_idx]:
                # Convert coordinates to integers for proper OpenCV pixel coordinates
                x1, y1 = int(x), int(y)
                x2, y2 = int(x + w), int(y + h)

                # Draw the bounding box (Bright Green)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

                # Draw the identity label (Yellow text on the bounding box header)
                label = f"ID: {obj_id}"
                cv2.putText(frame, label, (x1, max(y1 - 10, 0)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        # Write the rendered tracking frame to the output file stream
        out.write(frame)

    # 5. Clean up open writer resources
    out.release()
    print(f"Video visualization render complete! Output saved to: {output_video_path}\n")


# --- Execution Block for the Extracted Sequences ---
if __name__ == '__main__':
    # Map the exact output files and extracted folder paths used in your main tracker pipeline
    base_extraction_root = '/content'

    sequences_to_visualize = {
        '01': os.path.join(base_extraction_root, 'sequence_01'),
        '02': os.path.join(base_extraction_root, 'sequence_02'),
        '03': os.path.join(base_extraction_root, 'sequence_03'),
        '04': os.path.join(base_extraction_root, 'sequence_04')
    }

    # Run rendering loop across all your pipeline's tracking outputs
    for seq_id, extraction_dir in sequences_to_visualize.items():
        tracking_txt_path = f"{seq_id}.txt"       # Output text file generated by DeepSORT
        output_video = f"sequence_{seq_id}_visualized.mp4" # Visualized tracking file to save

        if os.path.exists(extraction_dir):
            create_tracking_video(extraction_dir, tracking_txt_path, output_video)
        else:
            print(f"Skipping visualization for sequence {seq_id}: Folder not found.")

Parsing tracking file: 01.txt...
Resolved sequence location: /content/sequence_01/01/img (429 frames found)
Generating video rendering: sequence_01_visualized.mp4...
Video visualization render complete! Output saved to: sequence_01_visualized.mp4

Parsing tracking file: 02.txt...
Resolved sequence location: /content/sequence_02/02/img (2782 frames found)
Generating video rendering: sequence_02_visualized.mp4...
Video visualization render complete! Output saved to: sequence_02_visualized.mp4

Parsing tracking file: 03.txt...
Resolved sequence location: /content/sequence_03/03/img (2405 frames found)
Generating video rendering: sequence_03_visualized.mp4...
Video visualization render complete! Output saved to: sequence_03_visualized.mp4

Parsing tracking file: 04.txt...
Resolved sequence location: /content/sequence_04/04/img (3315 frames found)
Generating video rendering: sequence_04_visualized.mp4...
Video visualization render complete! Output saved to: sequence_04_visualized.mp4

